# v3 模型改進：消融實驗

一次只多加一項改動，看每項改動各自的貢獻（test 都是同一批 325 張）：

| 版本 | 改動 | 資料 |
| --- | --- | --- |
| v2 | 基準（已完成） | 256 |
| **v3a** | + 厚度損失 + 強化資料增強 + 提早停止 | `echolvh_256_full.zip` |
| **v3b** | v3a + 解析度 384 | `echolvh_384_full.zip` |
| v3c | v3b + ResNet34（未執行：384 沒幫助） | `echolvh_384_full.zip` |
| **v3d** | **v3a + ResNet34** | `echolvh_256_full.zip` |

**使用方式**
1. 執行階段 → 變更執行階段類型 → **T4 GPU**
2. 下面第一格把 `RUN` 改成要跑的版本（`'v3a'`、`'v3b'`、`'v3c'` 或 `'v3d'`）
3. 從上到下一格一格執行。每個版本存在 `我的雲端硬碟/EchoLVH/checkpoints/<版本>/`，互不覆蓋
4. 跑完一個版本後，要跑下一版：**執行階段 → 中斷連線並刪除執行階段**，改 `RUN` 再從頭執行

中途斷線：從頭重新執行，會從 `last.pt` 接著訓練。

## 1. 選擇版本

In [ ]:
RUN = 'v3d'   # ← 可選 'v3a'、'v3b'、'v3c'、'v3d'

## 2. 確認 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. 連接 Google Drive、解壓資料、安裝套件

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/EchoLVH'
ZIP = {'v3a': 'echolvh_256_full.zip', 'v3b': 'echolvh_384_full.zip', 'v3c': 'echolvh_384_full.zip', 'v3d': 'echolvh_256_full.zip'}[RUN]
import os, glob, zipfile
src = f'{DRIVE_DIR}/{ZIP}'
assert os.path.exists(src), f'Drive 裡找不到 {ZIP}：請確認已上傳到「我的雲端硬碟/EchoLVH/」，且檔名完全相同（不能是「(1)」這種副本名）'
size_mb = os.path.getsize(src) / 1e6
print(f'{ZIP}：{size_mb:.0f} MB')
!cp "$src" /content/data.zip
try:
    with zipfile.ZipFile('/content/data.zip') as z:
        z.extractall('/content')
except zipfile.BadZipFile:
    raise SystemExit(f'zip 檔壞掉或還沒上傳完（目前 {size_mb:.0f} MB）。請到 Drive 刪掉重新上傳，等上傳完成再執行。')
os.remove('/content/data.zip')
n_img = len(glob.glob(f'/content/echolvh_{"384" if "384" in ZIP else "256"}/images/*.png'))
print('解壓出的圖片數：', n_img)
assert n_img > 10000, f'圖片只有 {n_img} 張（應該約 10381 張）：zip 可能不完整，請重新上傳'
!pip install -q segmentation-models-pytorch

## 4. 設定
- `DIST_W`：厚度損失的權重。訓練時除了「點的位置」，也直接比較「三段厚度」和醫師的差距（mm）
- `PATIENCE`：val 連續幾輪沒進步就停，避免過擬合、也省時間

In [ ]:
import os, json, math, random, time
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

# 三個實驗版本（消融實驗）：一次只多加一項改動
PRESETS = {
    'v3a': dict(zip='echolvh_256_full.zip', img=256, encoder='resnet18'),   # v2 + 厚度損失 + 強增強 + 提早停止
    'v3b': dict(zip='echolvh_384_full.zip', img=384, encoder='resnet18'),   # v3a + 384 解析度
    'v3c': dict(zip='echolvh_384_full.zip', img=384, encoder='resnet34'),   # v3b + ResNet34
    'v3d': dict(zip='echolvh_256_full.zip', img=256, encoder='resnet34'),   # v3a + ResNet34（384 沒幫助，改在 256 上試更強骨幹）
}
RUN = globals().get('RUN', 'v3a')          # ← 要跑哪一版就改這裡
P = PRESETS[RUN]
IMG, ENCODER = P['img'], P['encoder']
SIGMA = 2.0 * IMG / 256                     # 熱力圖光點大小隨解析度放大
BATCH = 16
EPOCHS = globals().get('EPOCHS', 30)       # 最多幾輪
PATIENCE = globals().get('PATIENCE', 6)    # val 連續幾輪沒進步就提早停止
LR = 1e-3
DIST_W = 0.2                                # 厚度損失的權重
DIST_WARMUP = 2                             # 前幾輪先只學位置，之後再加厚度損失
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KP = ['lvpw_outer', 'lvpw_lvid', 'lvid_ivs', 'ivs_outer']
SEGS = [('LVPWd', 0, 1), ('LVIDd', 1, 2), ('IVSd', 2, 3)]
DATA_DIR = globals().get('DATA_DIR', f'/content/echolvh_{IMG}')
CKPT_DIR = globals().get('CKPT_DIR', f'/content/drive/MyDrive/EchoLVH/checkpoints/{RUN}')
os.makedirs(CKPT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'RUN={RUN}  IMG={IMG}  ENCODER={ENCODER}  device={DEVICE}')

## 5. 資料（先全部讀進記憶體，約 1–3 分鐘）
強化版資料增強：旋轉 ±12°、縮放 0.85–1.15、平移、亮度對比、gamma、模糊、超音波斑點雜訊。

In [ ]:
from torch.utils.data import Dataset, DataLoader

labels = pd.read_csv(f'{DATA_DIR}/labels.csv')
if globals().get('SUBSET'):                 # 本機測試用
    labels = labels.groupby('split').head(SUBSET).reset_index(drop=True)

def load_all(df):
    """把圖片全部先讀進記憶體（約 1.5 GB），訓練時就不用每次讀硬碟，快很多。"""
    imgs, keep, missing, wrong = [], [], 0, None
    for i, r in enumerate(df.itertuples()):
        im = cv2.imread(f'{DATA_DIR}/images/{r.id}.png', cv2.IMREAD_GRAYSCALE)
        if im is None:
            missing += 1; continue
        if im.shape != (IMG, IMG):
            wrong = im.shape; continue
        imgs.append(im); keep.append(i)
        if (i + 1) % 2000 == 0: print(f'  已讀 {i+1}/{len(df)}')
    if not imgs:
        raise SystemExit(f'一張圖都沒讀到：{DATA_DIR}/images 裡缺 {missing} 張'
                         + (f'，或圖片大小是 {wrong} 而不是 {IMG}×{IMG}（RUN 和 zip 是否對應？）' if wrong else '')
                         + '。請重新執行第 3 格（解壓），或確認 Drive 上的 zip 完整。')
    if missing or wrong:
        print(f'⚠ 略過 {len(df) - len(keep)} 張讀不到或大小不對的圖')
    return df.iloc[keep].reset_index(drop=True), np.stack(imgs)

t0 = time.time()
labels, IMAGES = load_all(labels)
print(f'讀入 {len(labels)} 張（{time.time()-t0:.0f} 秒）', labels.split.value_counts().to_dict())

def augment(img, pts, sc_px):
    """強化版資料增強：幾何（旋轉、縮放、平移）+ 超音波常見的影像變化。點的座標同步轉換。"""
    ang = np.random.uniform(-12, 12); sc = np.random.uniform(0.85, 1.15)
    tx, ty = np.random.uniform(-16, 16, size=2) * IMG / 256
    M = cv2.getRotationMatrix2D((IMG / 2, IMG / 2), ang, sc); M[:, 2] += (tx, ty)
    img = cv2.warpAffine(img, M, (IMG, IMG), flags=cv2.INTER_LINEAR, borderValue=0)
    pts = (np.c_[pts, np.ones(4)] @ M.T).astype(np.float32)
    img = img * np.random.uniform(0.75, 1.25) + np.random.uniform(-0.1, 0.1)     # 亮度、對比
    img = np.clip(img, 0, 1) ** np.random.uniform(0.7, 1.4)                        # gamma
    if np.random.rand() < 0.3:                                                     # 模糊
        img = cv2.GaussianBlur(img, (0, 0), np.random.uniform(0.5, 1.2) * IMG / 256)
    if np.random.rand() < 0.5:                                                     # 斑點雜訊（speckle）
        img = img * (1 + np.random.normal(0, 0.12, img.shape).astype(np.float32))
    return np.clip(img, 0, 1).astype(np.float32), pts, sc_px / sc               # 縮放後每像素代表的公分也跟著變

class EchoKP(Dataset):
    def __init__(self, idx, train):
        self.idx, self.train = np.asarray(idx), train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; r = labels.iloc[j]
        img = IMAGES[j].astype(np.float32) / 255.0
        pts = np.array([[r[f'{k}_x'], r[f'{k}_y']] for k in KP], dtype=np.float32)   # 已是 IMG×IMG 上的座標
        cmpp = float(r.cm_per_px_256)                     # 欄位名沿用舊名：此解析度下每像素幾公分
        if self.train:
            img, pts, cmpp = augment(img, pts, cmpp)
        x = torch.from_numpy(np.repeat(img[None], 3, axis=0))
        return (x - 0.45) / 0.225, torch.from_numpy(pts), torch.tensor(cmpp, dtype=torch.float32)

split_idx = {s: np.where(labels.split.values == s)[0] for s in ['train', 'val', 'test']}
NW = globals().get('NUM_WORKERS', 2)
dl = {s: DataLoader(EchoKP(split_idx[s], s == 'train'), batch_size=BATCH, shuffle=(s == 'train'),
                    num_workers=NW, drop_last=(s == 'train'), persistent_workers=NW > 0)
      for s in ['train', 'val', 'test']}

In [ ]:
# 看幾張增強後的訓練圖，確認點還在心壁上
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
for a in ax:
    x, pts, _ = dl['train'].dataset[np.random.randint(len(dl['train'].dataset))]
    a.imshow(x[0], cmap='gray'); a.plot(pts[:, 0], pts[:, 1], 'r.-'); a.axis('off')
plt.show()

## 6. 模型、損失函數、TTA

In [ ]:
import segmentation_models_pytorch as smp

_ys = _xs = None
def grids(H, W, device):
    global _ys, _xs
    if _ys is None or _ys.shape[0] != H or _ys.device != device:
        _ys = torch.arange(H, device=device, dtype=torch.float32)
        _xs = torch.arange(W, device=device, dtype=torch.float32)
    return _ys, _xs

def make_heatmaps(pts, H, W):
    """在 GPU 上直接產生高斯熱力圖（比在 CPU 產生快很多）。pts: (B,K,2)"""
    ys, xs = grids(H, W, pts.device)
    gx = torch.exp(-(xs.view(1, 1, W) - pts[..., 0:1]) ** 2 / (2 * SIGMA ** 2))
    gy = torch.exp(-(ys.view(1, 1, H) - pts[..., 1:2]) ** 2 / (2 * SIGMA ** 2))
    return gy[..., :, None] * gx[..., None, :]

def to_prob(logits):
    B, K, H, W = logits.shape
    return logits.flatten(2).softmax(-1).view(B, K, H, W)

def soft_argmax(prob):
    """機率加權平均座標（可微分），用來計算厚度損失。"""
    B, K, H, W = prob.shape
    ys, xs = grids(H, W, prob.device)
    return torch.stack([(prob.sum(2) * xs).sum(-1), (prob.sum(3) * ys).sum(-1)], -1)

def decode(prob, win=2):
    """推論用：最亮點 + 周圍加權平均（次像素精度）。"""
    B, K, H, W = prob.shape
    idx = prob.flatten(2).argmax(-1)
    cy, cx = (idx // W).float(), (idx % W).float()
    ys, xs = grids(H, W, prob.device)
    my = ((ys.view(1, 1, H) - cy[..., None]).abs() <= win).float()
    mx = ((xs.view(1, 1, W) - cx[..., None]).abs() <= win).float()
    w = prob.clamp(min=0) * my[..., :, None] * mx[..., None, :]
    s = w.sum((-1, -2)).clamp(min=1e-12)
    return torch.stack([(w.sum(-2) * xs).sum(-1) / s, (w.sum(-1) * ys).sum(-1) / s], -1)

def seg_len_mm(pts, cmpp):
    """(B,4,2) 點 → (B,3) 三段長度（mm），順序 LVPWd, LVIDd, IVSd"""
    return torch.stack([(pts[:, b] - pts[:, a]).norm(dim=-1) for _, a, b in SEGS], -1) * cmpp[:, None] * 10

def losses(logits, pts, cmpp, use_dist):
    B, K, H, W = logits.shape
    gt = make_heatmaps(pts, H, W).flatten(2)
    gt = gt / gt.sum(-1, keepdim=True).clamp(min=1e-6)
    logp = logits.flatten(2).log_softmax(-1)
    ce = -(gt * logp).sum(-1).mean()                                   # 位置：空間 softmax 交叉熵
    if not use_dist:
        return ce, torch.zeros((), device=logits.device)
    pred = soft_argmax(logp.exp().view(B, K, H, W))
    dist = (seg_len_mm(pred, cmpp) - seg_len_mm(pts, cmpp)).abs().mean()  # 厚度：直接比較三段長度（mm）
    return ce + DIST_W * dist, dist

def build_model(pretrained=True):
    return smp.Unet(ENCODER, encoder_weights='imagenet' if pretrained else None, in_channels=3, classes=4)

# ---- TTA：把圖平移/縮放幾種版本各預測一次，座標換回原圖後取平均 ----
TTA = [(0, 0, 1.0), (4, 0, 1.0), (-4, 0, 1.0), (0, 4, 1.0), (0, -4, 1.0), (0, 0, 0.95), (0, 0, 1.05)]

def warp(x, dx, dy, s):
    """平移 (dx,dy) 像素並以中心縮放 s 倍（以 256 為基準，自動依解析度放大）"""
    B, C, H, W = x.shape
    tx, ty = 2 * dx * W / 256 / W, 2 * dy * H / 256 / H
    theta = torch.tensor([[1 / s, 0, -tx / s], [0, 1 / s, -ty / s]], dtype=x.dtype, device=x.device).expand(B, 2, 3)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, align_corners=False, padding_mode='zeros')

def unwarp_pts(p, dx, dy, s, H, W):
    """warp 後圖上的點 → 原圖座標"""
    tx, ty = 2 * dx * W / 256 / W, 2 * dy * H / 256 / H
    nx = ((2 * p[..., 0] + 1) / W - 1 - tx) / s
    ny = ((2 * p[..., 1] + 1) / H - 1 - ty) / s
    return torch.stack([((nx + 1) * W - 1) / 2, ((ny + 1) * H - 1) / 2], -1)

@torch.no_grad()
def predict(model, x, tta=False):
    if not tta:
        return decode(to_prob(model(x).float()))
    H, W = x.shape[-2:]
    outs = [unwarp_pts(decode(to_prob(model(warp(x, dx, dy, s)).float())), dx, dy, s, H, W) for dx, dy, s in TTA]
    return torch.stack(outs).median(0).values          # 用中位數，比平均更不怕單一離群預測

@torch.no_grad()
def evaluate(model, loader, tta=False):
    model.eval(); P_, T_, E_ = [], [], []
    for x, pts, cmpp in loader:
        pred = predict(model, x.to(DEVICE), tta).cpu()
        P_.append(seg_len_mm(pred, cmpp)); T_.append(seg_len_mm(pts, cmpp)); E_.append((pred - pts).norm(dim=-1))
    P_, T_, E_ = torch.cat(P_) / 10, torch.cat(T_) / 10, torch.cat(E_)   # cm
    mae = ((P_ - T_).abs() * 10).mean(0)
    return {'mae_mm': dict(zip([s[0] for s in SEGS], mae.tolist())), 'mean_mae_mm': mae.mean().item(),
            'point_err_px': E_.mean().item(), 'pred': P_.numpy(), 'true': T_.numpy()}

## 7. 訓練
每輪印出：總 loss、厚度誤差（訓練中）、val 三個量測的 MAE。v3b / v3c 每輪大約是 v3a 的 2 倍時間。

In [ ]:
model = build_model(pretrained=globals().get('PRETRAINED', True)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl['train']), pct_start=0.1)
scaler = torch.amp.GradScaler(enabled=(DEVICE == 'cuda'))
start, best, history, bad = 0, float('inf'), [], 0

if os.path.exists(f'{CKPT_DIR}/last.pt'):
    ck = torch.load(f'{CKPT_DIR}/last.pt', map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    start, best, history, bad = ck['epoch'] + 1, ck['best'], ck['history'], ck.get('bad', 0)
    print(f'從第 {start+1} 輪接著訓練（目前最佳 {best:.2f} mm）')

CONFIG = {'arch': 'Unet', 'encoder': ENCODER, 'img': IMG, 'keypoints': KP, 'run': RUN}
for ep in range(start, EPOCHS):
    if bad >= PATIENCE:
        print(f'val 已連續 {PATIENCE} 輪沒有進步 → 提早停止'); break
    model.train(); t0 = time.time(); tot = totd = 0
    use_dist = ep >= DIST_WARMUP
    for x, pts, cmpp in dl['train']:
        x, pts, cmpp = x.to(DEVICE), pts.to(DEVICE), cmpp.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            logits = model(x)
        loss, dist = losses(logits.float(), pts, cmpp, use_dist)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item(); totd += dist.item()
    v = evaluate(model, dl['val'])
    n = len(dl['train'])
    history.append({'epoch': ep, 'loss': tot / n, 'dist_mm': totd / n, 'val_mae_mm': v['mean_mae_mm'], **v['mae_mm']})
    flag = ''
    if v['mean_mae_mm'] < best:
        best, bad = v['mean_mae_mm'], 0; flag = '  ← 最佳，已存 best.pt'
        torch.save({'model': model.state_dict(), 'epoch': ep, 'val': v['mae_mm'], 'config': CONFIG}, f'{CKPT_DIR}/best.pt')
    else:
        bad += 1
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'epoch': ep,
                'best': best, 'history': history, 'bad': bad}, f'{CKPT_DIR}/last.pt')
    m = v['mae_mm']
    print(f"第 {ep+1:2d}/{EPOCHS} 輪 | loss {tot/n:.3f}（厚度 {totd/n:.2f} mm） | val IVSd {m['IVSd']:.2f} LVIDd {m['LVIDd']:.2f} "
          f"LVPWd {m['LVPWd']:.2f} mm | {time.time()-t0:.0f}s{flag}")

## 8. 訓練曲線

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoch + 1, h.loss, label='total'); ax[0].plot(h.epoch + 1, h.dist_mm, label='thickness (mm)')
ax[0].set_title('Train loss'); ax[0].legend(); ax[0].set_xlabel('epoch')
for c in ['IVSd', 'LVIDd', 'LVPWd']: ax[1].plot(h.epoch + 1, h[c], label=c)
ax[1].set_title('Val MAE (mm)'); ax[1].legend(); ax[1].set_xlabel('epoch'); plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/curves.png', dpi=120); plt.show()

## 9. 最終測試（有 / 沒有 TTA 各算一次）

In [ ]:
ck = torch.load(f'{CKPT_DIR}/best.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(ck['model'])
names = [s[0] for s in SEGS]

def summarize(t):
    P_, T_ = t['pred'], t['true']
    r2 = {n: float(1 - ((P_[:, i] - T_[:, i]) ** 2).sum() / ((T_[:, i] - T_[:, i].mean()) ** 2).sum()) for i, n in enumerate(names)}
    lt, lp = np.maximum(T_[:, 0], T_[:, 2]) >= 1.1, np.maximum(P_[:, 0], P_[:, 2]) >= 1.1
    return {'mae_mm': t['mae_mm'], 'mean_mae_mm': t['mean_mae_mm'], 'r2': r2, 'point_err_px': t['point_err_px'],
            'lvh_accuracy': float((lt == lp).mean()), 'lvh_sensitivity': float((lp & lt).sum() / max(lt.sum(), 1)),
            'lvh_specificity': float((~lp & ~lt).sum() / max((~lt).sum(), 1))}

t_plain, t_tta = evaluate(model, dl['test']), evaluate(model, dl['test'], tta=True)
result = {'run': RUN, 'img': IMG, 'encoder': ENCODER, 'best_epoch': int(ck['epoch']) + 1, 'n_test': int(len(t_plain['pred'])),
          'no_tta': summarize(t_plain), 'tta': summarize(t_tta)}
json.dump(result, open(f'{CKPT_DIR}/test_result.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(result, indent=2, ensure_ascii=False))

## 10. 散佈圖與範例

In [ ]:
P_, T_ = t_tta['pred'], t_tta['true']
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for i, n in enumerate(names):
    ax[i].scatter(T_[:, i], P_[:, i], s=8, alpha=0.6)
    lo, hi = T_[:, i].min(), T_[:, i].max(); ax[i].plot([lo, hi], [lo, hi], 'r--', lw=1)
    ax[i].set_title(f"{n}  MAE {t_tta['mae_mm'][n]:.2f} mm (TTA)"); ax[i].set_xlabel('Doctor (cm)'); ax[i].set_ylabel('Model (cm)')
plt.tight_layout(); plt.savefig(f'{CKPT_DIR}/test_scatter.png', dpi=120); plt.show()

model.eval(); ds = dl['test'].dataset
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, i in zip(ax.flat, np.random.RandomState(0).choice(len(ds), 8, replace=False)):
    x, pts, cmpp = ds[i]
    pred = predict(model, x[None].to(DEVICE), tta=True).cpu()[0]
    a.imshow(x[0], cmap='gray'); a.plot(pts[:, 0], pts[:, 1], 'g.-'); a.plot(pred[:, 0], pred[:, 1], 'r.-'); a.axis('off')
plt.suptitle('green = doctor, red = model'); plt.tight_layout(); plt.savefig(f'{CKPT_DIR}/test_examples.png', dpi=110); plt.show()

## 11. 消融實驗總表（自動讀取所有已完成的版本）

In [ ]:
rows = []
def add(name, path, key=None):
    if not os.path.exists(path): return
    j = json.load(open(path)); r = j[key] if key else j
    rows.append({'版本': name, **{f'{k} MAE (mm)': round(v, 2) for k, v in r['mae_mm'].items()},
                 '平均 MAE (mm)': round(sum(r['mae_mm'].values()) / 3, 2),
                 '肥厚判斷準確率': f"{r['lvh_accuracy']:.1%}"})
base = '/content/drive/MyDrive/EchoLVH/checkpoints'
add('v2（基準）', f'{base}/v2_full/test_result.json')
for v in ['v3a', 'v3b', 'v3c', 'v3d']:
    add(f'{v}', f'{base}/{v}/test_result.json', 'no_tta')
    add(f'{v} + TTA', f'{base}/{v}/test_result.json', 'tta')
table = pd.DataFrame(rows); display(table)
table.to_csv(f'{base}/ablation_table.csv', index=False, encoding='utf-8-sig')
print('已存：checkpoints/ablation_table.csv')

## 完成！
把 `checkpoints/<版本>/test_result.json`（或最後一格的 `ablation_table.csv`）傳給 Claude。
最好的版本的 `best.pt` 之後要放到電腦的 `EchoLVH/checkpoints/` 給後端用。